# 🩺 MediRAG AI - Google Colab High-Speed Document Indexer

This notebook extracts the medical reference PDF (`Medical_book.pdf`), chunks it, computes dense embeddings on **GPU (CUDA)** using Hugging Face, and uploads them to **Weaviate Cloud (WCD)**.

> **GPU Setup:** Make sure to select **Runtime** ➔ **Change runtime type** ➔ **T4 GPU** for 10x faster embeddings.

### Step 1: Install Dependencies

In [ ]:
!pip install -q -U langchain langchain-core langchain-huggingface langchain-text-splitters langchain-weaviate weaviate-client sentence-transformers pypdf python-dotenv

### Step 2: Upload or Locate `Medical_book.pdf`
Create a `data` folder on the left file panel and upload `Medical_book.pdf` into it.

In [ ]:
import os

os.makedirs('data', exist_ok=True)

if os.path.exists('data/Medical_book.pdf'):
    print("✅ Found data/Medical_book.pdf!")
else:
    print("⚠️ Please upload Medical_book.pdf into the 'data/' folder on the left file panel.")

### Step 3: Enter Weaviate Cloud Credentials

In [ ]:
import getpass

raw_url = input("Enter your Weaviate Cloud REST URL (e.g. csny7zj5...weaviate.cloud): ").strip()
WEAVIATE_URL = raw_url if raw_url.startswith("http") else f"https://{raw_url}"
WEAVIATE_API_KEY = getpass.getpass("Enter your Weaviate API Key (hidden): ").strip()

print(f"✅ Configured URL: {WEAVIATE_URL}")

### Step 4: Extract & Chunk PDF (Standalone pypdf Loader)

In [ ]:
import os
import glob
from typing import List
from pypdf import PdfReader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("Loading PDF document from data/ using standalone pypdf...")
documents: List[Document] = []
pdf_files = glob.glob('data/*.pdf')

for file_path in pdf_files:
    reader = PdfReader(file_path)
    print(f"Processing {file_path} ({len(reader.pages)} pages)...")
    for page_idx, page in enumerate(reader.pages):
        text = page.extract_text()
        if text and text.strip():
            documents.append(Document(page_content=text, metadata={"source": file_path, "page": page_idx}))

print(f"Extracted {len(documents)} document pages.")

# Split into 500-character chunks with 20-character overlap
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
text_chunks = text_splitter.split_documents(documents)
print(f"Generated {len(text_chunks)} total text chunks.")

### Step 5: Initialize Embeddings Model (GPU-Accelerated)

In [ ]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device} (GPU accelerated: {device == 'cuda'})")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 64}
)

### Step 6A: Quick Test — Ingest 5 Chunks First
Run this cell first to test connection and verify vector storage in Weaviate Cloud.

In [ ]:
import weaviate
from weaviate.auth import Auth
from langchain_weaviate import WeaviateVectorStore

client = weaviate.connect_to_weaviate_cloud(
    cluster_url=WEAVIATE_URL,
    auth_credentials=Auth.api_key(WEAVIATE_API_KEY)
)

index_name = "MediragAiDb"

# 1. Delete previous/broken collection if it exists to ensure fresh vector indexing
if client.collections.exists(index_name):
    print(f"Deleting old collection '{index_name}'...")
    client.collections.delete(index_name)

# 2. Create the collection allowing the server to use its default vector index type (hfresh)
print(f"Creating collection '{index_name}'...")
client.collections.create(
    name=index_name,
    vectorizer_config=None  # We generate embeddings locally using HuggingFace
)

print(f"Testing ingestion with 5 chunks into '{index_name}'...")
sample_chunks = text_chunks[:5]

try:
    # Initialize WeaviateVectorStore using the already created collection
    docsearch = WeaviateVectorStore(
        client=client,
        index_name=index_name,
        embedding=embeddings,
        text_key="text"
    )

    # Add the test documents
    docsearch.add_documents(sample_chunks)
    print("🎉 SUCCESS: 5 test chunks successfully stored in Weaviate Cloud!")

    # Test search query
    test_results = docsearch.similarity_search("What is asthma?", k=1)
    if test_results:
        print("\n--- Verification Search Result ---")
        print(test_results[0].page_content[:200] + "...")
finally:
    client.close()

### Step 6B: Full Ingestion of All 5,861 Chunks (in Batches)
Run this cell after Step 6A succeeds to ingest the entire medical textbook with batch progress tracking.

In [ ]:
import weaviate
from weaviate.auth import Auth
from langchain_weaviate import WeaviateVectorStore

client = weaviate.connect_to_weaviate_cloud(
    cluster_url=WEAVIATE_URL,
    auth_credentials=Auth.api_key(WEAVIATE_API_KEY)
)

index_name = "MediragAiDb"

# Clear collection for complete, clean dataset
if client.collections.exists(index_name):
    print(f"Clearing '{index_name}' for full ingestion...")
    client.collections.delete(index_name)

# Create fresh collection with default vectorizer_config=None
print(f"Creating fresh collection '{index_name}'...")
client.collections.create(
    name=index_name,
    vectorizer_config=None
)

batch_size = 200
total_chunks = len(text_chunks)
total_batches = (total_chunks + batch_size - 1) // batch_size

try:
    docsearch = WeaviateVectorStore(
        client=client,
        index_name=index_name,
        embedding=embeddings,
        text_key="text"
    )

    print(f"Ingesting all {total_chunks} chunks in batches of {batch_size}...")
    for i in range(0, total_chunks, batch_size):
        batch = text_chunks[i:i + batch_size]
        batch_num = (i // batch_size) + 1
        print(f"Uploading Batch {batch_num}/{total_batches} ({len(batch)} chunks)...")
        docsearch.add_documents(batch)
            
    print("🎉 ALL CHUNKS INGESTED TO WEAVIATE CLOUD SUCCESSFULLY!")
finally:
    client.close()

### Step 7: Final Query Test
Test search accuracy directly from Weaviate Cloud.

In [ ]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=WEAVIATE_URL,
    auth_credentials=Auth.api_key(WEAVIATE_API_KEY)
)

try:
    test_search = WeaviateVectorStore(
        client=client,
        index_name="MediragAiDb",
        text_key="text",
        embedding=embeddings
    )
    
    query = "What are the common symptoms of diabetes?"
    print(f"Query: {query}\n")
    results = test_search.similarity_search(query, k=3)
    for i, doc in enumerate(results):
        print(f"=== Result {i+1} (Source: {doc.metadata.get('source')}) ===")
        print(doc.page_content.strip())
        print()
finally:
    client.close()